# Práctica 2. Hadoop MapReduce.

## SESION 1. Manejo de un cluster Hadoop v3. Gestión HDFS y YARN.

**IMPORTANTE: Recuerda apagar el sistema docker cuando acabes de trabajar para que no se quede inconsistente (script stop.sh)**  
**HDFS es un sistema de archivos append-only (solo añadir) diseñado para ser inmutable, NO intentes editar ficheros montandos en `~/hdfs`, ahí sólo debes copiar y borrar. Si necesitas editar algo, sacalo fuera, borralo y vuelve a subirlo modificado.**

En esta práctica utilizaremos contenedores Docker para trabajar sobre un cluster Hadoop. Si no has usado docker nunca, aquí hay un video corto pero muy detallado de como funciona (8 minutos en ingles) https://youtu.be/JSLpG_spOBM

Debido a que la instalación y despliegue de un cluster Hadoop no es objetivo formativo de esta asignatura, los profesores proporcionan un fichero de configuración para docker compose a través del repositorio público de la asignatura (https://github.com/ProcParDatos/ppd-public) dentro de la carpeta environment/p2, así como una serie de scripts para instalar docker (install_docker.sh), arrancar el sistema (start.sh), detenerlo (stop.sh) y limpiar los contenedores docker (cleanup.sh). Si tienes curiosidad puedes echarle un vistazo para ver que hacen.

Una vez arrancado el sistema se montará la carpeta de hdfs vía nfs para facilitar el trabajo.   

**Para detalles sobre la instalación, montaje y pruebas, comprobad el fichero "environment/p2/README.md"**

Tomando como punto de partida este sistema, realizaremos diversas actividades prácticas para familiarizarnos con el manejo del sistema de ficheros HDFS y la gestión de recursos basada en YARN.

**ENTREGA OBLIGATORIA**

Una de las actividades de la Tarea 2 de la asignatura (se publicará próximamente) requiere que el alumno demuestre haber seguido los pasos de este notebook. Con este propósito, se solicita generar una memoria que incluya capturas de pantalla, explicaciones y todos los detalles que el alumno considere necesarios para demostrar que ha realizado los pasos y entendido las preguntas y/o tareas que se formulan en las siguientes secciones.

**NOTA IMPORTANTE**

El documento no puede consistir en una mera secuencia de imágenes. Es necesario describir los comandos ejecutados, interpretar la información que se obtiene y razonar la manera de resolver las tareas propuestas.


---
### APARTADO 1. Primeros pasos. Descripción del escenario.

1. Sigue las instrucciones para instalar docker, arrancar los servicios con docker compose y conectar el sistema de ficheros HDFS a tu home (~/hdfs) vía NFS (`sudo ./mount-hdfs.sh mount`). Para desmontar antes de apagar docker usa `sudo ./mount-hdfs.sh umount`

2. Los contenedores docker representan una serie de nodos en un "cluster" virtual, que incluye: 

    - namenode: gestiona el sistema de ficheros HDFS (metadatos, namespace, ubicación de bloques) y coordina los DataNodes. En este entorno también corre ResourceManager de YARN.
    - datanode1, datanode2, datanode3: almacenan los bloques reales de los ficheros y atienden lecturas/escrituras/replicación. En este entorno también ejecutan NodeManager.
    - nfsgateway: publica HDFS por NFSv3 para poder montarlo como carpeta local (~/hdfs) desde el host.
    - historyserver: guarda y muestra el histórico de jobs MapReduce terminados (UI de JobHistory).
    - workbench: nodo cliente para lanzar comandos hdfs/yarn, pruebas y jobs; no es nodo de almacenamiento de HDFS.

<div style="text-align: center;">
<img src="data:image/png;base64,
...
" width="500"/>
</div>

##### 1.1.5 Acceso a los interfaces web de los demonios

Los diferentes servicios de Hadoop ofrecen una interfaz web de control. Una vez arrancado el entorno docker, abre un navegador y comprueba los siguientes enlaces:

- http://localhost:9870 interfaz web del HDFS NameNode.
- http://localhost:8088 interfaz web de YARN.
- http://localhost:19888 interfaz web del historial de trabajos (JobHistory) 


---
### APARTADO 2: Inicio del cluster Hadoop con contenedores Docker

Como hemos comentado, se proporciona un fichero YAML para docker compose con toda la configuración del cluster HDFS (docker-compose.yml) para levantarlo con un sólo comando.

1. Inicia el cluster con el script o manualmente 

```bash
start.sh
```

2. Los contenedores en ejecución pueden consultarse con el comando `docker container ps`. Comprueba que están ejecutándose correctamente y anota la salida.

3. Conéctate al **NameNode/Resource Manager** ejecutando:

```bash
docker container exec -ti namenode /bin/bash
```

   y una vez dentro de contenedor, cambiar al usuario `hdadmin`

```bash
su - hdadmin
```

3. Todos los demonios de Hadoop se configuran, principalmente, mediante cuatro ficheros, localizados en `$HADOOP_HOME/etc/hadoop/`, en los que se pueden indicar un gran número de propiedades (véase http://hadoop.apache.org/docs/stable3/hadoop-project-dist/hadoop-common/ClusterSetup.html para más información):

- `core-site.xml`: configuración principal, valores por defecto en http://hadoop.apache.org/docs/stable3/hadoop-project-dist/hadoop-common/core-default.xml

- `hdfs-site.xml`: configuración del HDFS, valores por defecto en http://hadoop.apache.org/docs/stable3/hadoop-project-dist/hadoop-hdfs/hdfs-default.xml

- `yarn-site.xml`: configuración del YARN, valores por defecto en http://hadoop.apache.org/docs/stable3/hadoop-yarn/hadoop-yarn-common/yarn-default.xml

- `mapred-site.xml`: configuración del MapReduce, valores por defecto en http://hadoop.apache.org/docs/stable3/hadoop-mapreduce-client/hadoop-mapreduce-client-core/mapred-default.xml

Localiza estos ficheros y revisa su contenido (comando `cat` por ejemplo). Te será fácil interpretar los parámetros de configuración ya que estos se encuentran comentados. No es necesirio mostrar el contenido de estos ficheros en la memoria.

4. Podemos comprobar la información de estado del cluster de manera más humana interactuando directamente con cada servicio:

```bash
hdfs dfsadmin -report
yarn node -list
```

  Analiza la información que devuelve cada uno de los dos comandos anteriores. ¿Qué conclusiones puedes extraer? ¿Qué parte es la más relevante a la hora de determinar si el cluster Hadoop está funcionando correctamente?

5. Los diferentes servicios de Hadoop ofrecen una interfaz web de control. Concretamente, los demonios que implementan la funcionalidad de NameNode y ResourceManagner ofrecen una interfaz web que podemos acceder a través de los siguientes enlaces:

- http://localhost:9870 interfaz web del HDFS NameNode.
- http://localhost:8088 interfaz web de YARN.
- http://localhost:19888 interfaz web del historial de trabajos (JobHistory) 

Abre el navegador y comprueba que puedes acceder a todas las interfaces web. Analiza la información que aparece y compárala con la obtenida en el punto anterior mediante los comandos `hdfs dfsadmin -report` y `yarn node -list`.

6. Para salir del contenedor hay que escribir `exit` dos veces: una saldrá del usuario `hdadmin` de vuelta hacia `root`, y el último `exit` saldrá del `exec` hacia el contenedor (aunque el contenedor continúa ejecutándose en segundo plano):

```bash
hdadmin@namenode:~$ exit
logout
root@namenode:/# exit
exit
```
<br>

**Por seguridad, antes de apagar tu sistema, debes PARAR TODOS los contenedores** (no lo hagas ahora porque vas a seguir necesitándolos). Para ello bastará con ejecutar el comando `docker compose down` o el script `stop.sh` 

```bash
stop.sh
```

Cuando detienes los contenedores, si ejecutas el comando `docker compose ps` verás que devuelve una lista vacía. Esto se debe a que este comando sólo muestra los contenedores en ejecución.

Cuando quieras volver a iniciar los contenedores, vuelve a ejecutar el script `start.sh` o el comando `docker compose up -d --build`

```bash
start.sh
```

<span style="color: red;">**RECUERDA, por seguridad, SIEMPRE debes PARAR los contenedores ANTES DE APAGAR tu máquina**</span>


---
### APARTADO 3: Interfaz línea de comandos para acceder al sistema de ficheros HDFS. Creación de directorios y copia de datos.

####  3.1 Creación de los directorios de los usuarios en HDFS

Dentro del workbench (`docker compose exec workbench bash`), comprueba que carpetas hay para cada usuario 

```bash
hdfs dfs -ls /user
```

Crea un directorio de prueba y un fichero de prueba.  
Comprueba quien es el usuario dentro del hdfs que estamos utilizando por defecto desde el contenedor "workbench":

```bash
hdfs dfs -mkdir -p prueba_dir
hdfs dfs -touch prueba_file
hdfs dfs -ls 
```

Abre una nueva terminal (wsl/ubuntu) sin cerrar la anterior. Asegurate que has montado HDFS vía NFS (`sudo ./mount-hdfs.sh mount`).  
Ve a la carpeta ~/hdfs. Comprueba si puedes ver los ficheros/carpetas que creaste con `hdfs dfs`.
 
```bash
cd ~/hdfs
ls -la 
```

Y crea un fichero y una carpeta desde esta terminal

```bash
touch prueba_file_2
mkdir prueba_dir_2
```

¿Que diferencias observas con la información de usuario y grupo entre las dos terminales después de crear los dos ficheros y dos carpetas (`hdfs dfs -ls` vs `ls -la`)? 
¿a que se debe el código numérico que aparece en el grupo en algunos de los ficheros? (puedes buscar info online)

<span style="color: green;">**PISTA**</span>: para ver el mapeo que tenemos configurado en nuestro entorno entre los identificadores del usuario local y el por defecto de hdfs (luser):  
```bash
docker compose exec nfsgateway id -u luser
docker compose exec nfsgateway id -g luser
tail /etc/passwd | grep $USER
```


#### 3.2 Copia de ficheros de usuario


1. Tener el sistema de ficheros montado vía NFS nos facilita mover los ficheros hacia y desde HDFS como si fuera parte de nuestro sistema de ficheros local. Descarga el fichero https://github.com/dsevilla/bd2-data/releases/download/libros-tar-gz/libros.tar.gz que contiene unos ficheros de ejemplo que usaremos en esta practica desde la terminal fuera del contenedor (en tu home por ejemplo):

```bash
cd $HOME
wget https://github.com/dsevilla/bd2-data/releases/download/libros-tar-gz/libros.tar.gz
tar xzf libros.tar.gz
rm libros.tar.gz
```

2. Copia los libros a la carpeta `~/hdfs`

```bash
cp -r libros ~/hdfs
```

Comprueba que se han copiado desde la terminal dentro del contendor "workbench"

```bash
hdfs dfs -ls libros
```

3. Observa que hay un fichero que es significativamente más grande que los demás, y que supera el tamaño de bloque que tenemos configurado (64MB) (puedes comprobarlo con `hdfs getconf -confKey dfs.blocksize`).  
Vamos a obtener detalles sobre este fichero vía terminal:

```bash
hdfs fsck libros/random_words.txt.bz2
```

Si te aparece un mensaje como:

```bash
Total size:    0 B (Total open files size: 330326458 B) 
Total files:   0 (Files currently being written: 1)
```

Es que se han quedado los streams abiertos desde tu sistema local al nfsgateway de docker, espera unos minutos y si sigue pasando puedes forzar el cierre ejecutando el script `recover-open-leases.sh`.

Desde el interfaz gráfico de HDFS también podemos ver información sobre los ficheros y donde se encuentran las replicas de sus bloques 
   - Abre el navegador y conéctate al interfaz web del NameNode (http://localhost:9870)
   - Ve al menu "Utilities" -> "Browse the filesystem"
   - Localiza el fichero  **random_words.txt.bz2**. Comprueba y anota cuántos bloques se emplean para almacenar el contenido de este fichero y cuantas replicas tiene cada bloque.


---
### APARTADO 4. Interfaz Python para acceder al sistema de ficheros HDFS.

#### 4.1 Ejecución del código `filesystem_cat` (ver transparencias tema 2)


1. Desde la terminal fuera de docker, copia el fichero `filesystem_cat.py` al nodo de trabajo (workbench) y crea un par de ficheros de prueba en el HDFS.

```bash   
docker cp $HOME/ppd-public/practicas/p2/2_1_hadoop/filesystem_cat.py workbench:/home/luser
echo "HDFS es el sistema de archivos que permite a Hadoop confiar en hardware poco fiable" > $HOME/hdfs/fichero1.txt
```

2. Si no tienes ya una terminal abierta dentro del workbench, abrela:

```bash
docker compose exec workbench bash
```
   
3. Para ejecutar el script necesitamos el paquete pyarrow, el cual ya viene instalado en workbench mediante el Dockerfile.  
   Lo ejecutaremos como usuario `luser` ya que los ficheros los copiamos al HDFS de dicho usuario.  
   También necesitamos establecer correctamente la variable de entorno CLASSPATH:  
    ```bash
    su - luser
    export CLASSPATH=$(hdfs classpath --glob)
    ```

4. Ahora vamos a ver si podemos ver el contenido de un fichero de texto (sin comprimir) que esté subido a HDFS.
En el punto 1 creamos dos ficheros de prueba (`fichero1.txt` y `fichero2.txt`):

```bash
# python3 filesystem_cat.py hdfs://namenode:9000/ruta/completa/fichero/a/mostrar
python3 filesystem_cat.py hdfs://namenode:9000/user/luser/fichero1.txt
```

#### 4.2 Tarea: implementa el programa `copy_half_file`


1. Basándote en el código anterior y en la plantilla de programa `ppd-public/practicas/p2/2_1_hadoop/copy_half_file.py`, escribe un programa que copie la mitad inferior del fichero en HDFS a otro fichero en otro directorio del HDFS  


2. Pistas:
   - Usa `get_file_info()` para obtener la longitud del fichero.
   - Para moverte a la mitad de un fichero, usa `seek()`.
   - Para realizar la copia, usa la biblioteca *shutil*.

---
### APARTADO 5: Prueba de aplicaciones MapReduce simples

#### 5.1 Aplicación MapReduce para el cálculo de Pi

Vamos a probar nuestra primera aplicación MapReduce haciendo uso de un ejemplo que viene incluido en la instalación Hadoop. Este ejemplo calcula una aproximación del valor de π (pi) utilizando un enfoque de Monte Carlo. El proceso funciona de la siguiente manera:

1. **Generación de puntos aleatorios**:
   - Cada *tarea map* genera un conjunto de puntos aleatorios en un cuadrado de 2x2 (con coordenadas entre -1 y 1).
   - Estos puntos se generan dentro de un círculo inscrito en el cuadrado (de radio 1).

2. **Conteo de puntos dentro del círculo**:
   - Para cada punto generado, se verifica si está dentro del círculo (usando la fórmula $ x^2 + y^2 \leq 1 $).
   - Si el punto está dentro del círculo, se cuenta como un "éxito".

3. **Cálculo de la proporción**:
   - La proporción de puntos dentro del círculo respecto al total de puntos generados se usa para aproximar el valor de π.
   - Dado que el área del círculo es $\pi r^2 $ y el área del cuadrado es $4$, la proporción de puntos dentro del círculo es aproximadamente $\frac{\pi}{4}$.
   - Por lo tanto, $\pi \approx 4 \times \frac{\text{número de puntos dentro del círculo}}{\text{número total de puntos}}$.

4. **Agregación de resultados**:
   - Los resultados de todos las *tareas map* se combinan en la *fase de reducción* para obtener un conteo total de puntos dentro del círculo y un conteo total de puntos generados.
   - Finalmente, se calcula el valor de π usando la fórmula anterior.


En el nodo de workbench, **como usuario `luser`**, ejecuta los siguientes comandos para lanzar la aplicación MapReduce que implementa la aproximación de Pi empleando el proceso descrito anteriormente:

```bash
docker compose exec workbench bash
su - luser
export MAPRED_EXAMPLES=$HADOOP_HOME/share/hadoop/mapreduce
yarn jar $MAPRED_EXAMPLES/hadoop-mapreduce-examples-*.jar pi 16 1000
```

Monitoriza la ejecución de esta tarea por medio del interfaz web de Yarn. Explora todas las opciones y menus disponibles e identifica toda la información que resume la ejecución de la tarea MapReduce.

#### 5.2 Aplicación WordCount

Ahora vamos a probar una aplicación MapReduce que implementa el ejemplo de conteo de palabras estudiado en clase de teoría. Revisa las transparencias para recordar como funciona.

1. Copia el código en al nodo workbench al usuario `luser`

```bash
docker cp $HOME/ppd-public/practicas/p2/2_1_hadoop/wordcount.tgz namenode:/home/luser
```

2. Conéctate al nodo workbench y, como usuario `luser`, descomprímelo y compílalo usando maven:
    
```bash
docker compose exec workbench bash
su - luser
cd; tar xvzf wordcount.tgz
cd wordcount
mvn package
```

3. Ejecútalo con el comando Yarn:

```bash
yarn jar target/wordcount*.jar libros/p* wordcount-out
```

  Comprueba en el interfaz web de Yarn la ejecución de esta tarea.

4. Trae los ficheros de salida del HDFS al disco local del workbench:
    
```bash
hdfs dfs -get wordcount-out
```

5. Analiza la salida que ha generado la ejecución de la tarea MapReduce. ¿Porqué se generan diversos ficheros? (Pista, mira el fichero `wordcount/src/main/java/cursohadoop/wordcount/WordCountDriver.java`)

6. Analiza la información de salida que devuelve el comando yarn y constrastala con la información disponible a través de la web YARN. ¿Cuántos splits se han generado? ¿Cuántas tareas map se han ejecutado? ¿Cuántas tareas reduce? ¿Falló la ejecución de alguna tareas map y o reduce? ¿Cómo ha repartido la función de particionado los pares (clave,valor) intermedios? (consulta el servidor de JobHistory, selecciona tu aplicacion, luego haz click en las tareas reduce para ver la lista, selecciona una y en el menu lateral puedes seleccionar la sección "Counters" para ver sus "input groups" y compararlos con las otras tareas reduce).

7. Comprueba también los ficheros de salida para entender mejor la salida. 

```bash
cd wordcount-out
tail part-r-00000
tail part-r-00001
tail part-r-00002
```

¿Ha repartido de manera uniforme las claves enviadas a los reducers?
